# Défi Quotidien : Attention Multi-Têtes et Comparaisons de Transformers

## 👩‍🏫 👩🏿‍🏫 Ce que nous allons apprendre

Dans ce défi, nous allons explorer comment le mécanisme d'attention fonctionne au sein des architectures de transformeurs. Nous implémenterons nos propres modules d'attention multi-têtes. À la fin, nous devrions être capables de :

*   Expliquer les différences entre l'attention à tête unique (single-head), l'attention multi-têtes (multi-head) et l'attention croisée (cross-attention).
*   Implémenter un bloc d'attention par produit scalaire mis à l'échelle (scaled dot-product attention) et l'étendre pour créer une attention multi-têtes.
*   Comparer un encodeur d'attention personnalisé à un transformeur pré-entraîné (comme DistilBERT ou BERT).
*   Analyser les cartes d'attention pour comprendre sur quoi le modèle se concentre.
*   Évaluer et réfléchir aux compromis entre des piles d'attention personnalisées légères et de grands modèles pré-entraînés.


## 🛠️ Ce que nous allons créer

Nous allons produire :

*   Un module PyTorch personnalisé implémentant l'attention multi-têtes et les blocs d'encodeur à réseau de neurones (feedforward).
*   Une base de référence de transformeur *fine-tunée* (ajustée) sur le même ensemble de données.
*   Des visualisations des poids d'attention pour des échantillons sélectionnés.
*   Une réflexion comparant les deux approches et documentant les aperçus sur le comportement de l'attention.

## Données

Nous utiliserons l'ensemble de données d'inférence en langage naturel (Natural Language Inference, NLI) fourni via un lien (que nous intégrerons plus tard si nous faisons la partie entraînement).

## Tâche : Implémentation de l'Attention à Tête Unique (Single-Head Attention)

**Objectif :** Implémenter le bloc de construction de base avant de l'étendre à plusieurs têtes.

**Instructions :**
*   En utilisant PyTorch, implémentez un module d'Attention avec des projections linéaires pour Q (Query), K (Key) et V (Value).
*   Validez les formes (shapes) avec des tenseurs factices (batch, seq_len, hidden_dim).
*   Enregistrez les poids d'attention pour inspection.
*   Fonctions à utiliser : `torch.matmul`, `torch.softmax`, `torch.nn.Linear`.

---

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Voici notre première classe ! Nous allons créer le composant de base de l'attention.
# C'est l'attention par produit scalaire mise à l'échelle, où nous calculons comment les mots sont liés entre eux.
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_model):
        super(ScaledDotProductAttention, self).__init__()
        # d_model est la dimension de notre modèle, la taille de nos vecteurs de mots (embeddings).
        self_d_k = d_model # Nous utilisons d_model pour la dimension de la clé (K) et de la requête (Q).

        # Nous avons besoin de couches linéaires pour transformer nos entrées en Q, K et V.
        # Imaginez que Q, K, V sont différentes 'perspectives' du même mot.
        self.query_projection = nn.Linear(d_model, self_d_k)
        self.key_projection = nn.Linear(d_model, self_d_k)
        self.value_projection = nn.Linear(d_model, d_model) # V garde la même dimension de sortie pour l'agrégation.

    def forward(self, query, key, value, mask=None):
        # 1. Projeter Q, K, V avec nos couches linéaires.
        # Cela transforme nos entrées originales en vecteurs Q, K, V spécifiques à l'attention.
        q = self.query_projection(query)  # shape: (batch_size, seq_len, d_k)
        k = self.key_projection(key)      # shape: (batch_size, seq_len, d_k)
        v = self.value_projection(value)  # shape: (batch_size, seq_len, d_model)

        # Récupérons la dimension d_k pour la mise à l'échelle.
        # C'est important pour que nos scores d'attention ne deviennent pas trop grands.
        d_k = q.size(-1)

        # 2. Calculer les scores d'attention.
        # C'est le cœur de l'attention : nous mesurons la similarité entre chaque Q et chaque K.
        # torch.matmul(q, k.transpose(-2, -1)) c'est (Q * K^T).
        # Le .transpose(-2, -1) échange les deux dernières dimensions de K.
        scores = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

        # 3. Appliquer un masque (si fourni).
        # Un masque est utile pour ignorer certaines parties de la séquence (par exemple, les jetons de 'padding').
        if mask is not None:
            # Nous mettons des valeurs très petites (-1e9) là où le masque est vrai (0) pour qu'elles aient un poids presque nul après softmax.
            scores = scores.masked_fill(mask == 0, -1e9)

        # 4. Appliquer Softmax pour obtenir les poids d'attention.
        # Softmax transforme les scores en probabilités, où la somme de chaque ligne est de 1.
        attention_weights = F.softmax(scores, dim=-1)

        # 5. Multiplier les poids d'attention par V.
        # C'est ici que nous agrégeons les informations : chaque V est pondéré par l'attention que Q lui porte.
        output = torch.matmul(attention_weights, v)

        # Nous retournons l'output (le résultat de l'attention) et les poids d'attention (pour l'analyse).
        return output, attention_weights


# --- Validation des formes avec des tenseurs factices ---
print("\n--- Validation de ScaledDotProductAttention ---")
# Paramètres de nos tenseurs factices (imaginez des données réelles).
batch_size = 2      # Nombre d'exemples dans notre lot.
seq_len = 5         # Longueur de la séquence (nombre de mots).
hidden_dim = 64     # Dimension de nos vecteurs de mots (embeddings).

# Créons des tenseurs d'entrée factices.
# Ces tenseurs simulent l'entrée que notre modèle recevrait.
# Dans le cas de l'auto-attention, Q, K, V sont souvent les mêmes tenseurs d'entrée.
dummy_input = torch.randn(batch_size, seq_len, hidden_dim)

print(f"Forme de l'entrée factice : {dummy_input.shape}")

# Instancions notre module d'attention.
attention_module = ScaledDotProductAttention(hidden_dim)

# Effectuons une passe avant (forward pass) avec nos tenseurs factices.
output, weights = attention_module(dummy_input, dummy_input, dummy_input)

print(f"Forme de la sortie de l'attention : {output.shape}")
print(f"Forme des poids d'attention : {weights.shape}")

# Vérifions que les formes sont comme prévu :
# La sortie devrait avoir la même forme que V : (batch_size, seq_len, hidden_dim)
# Les poids d'attention devraient être (batch_size, seq_len, seq_len)
# Chaque ligne des poids d'attention (sommée sur la dernière dimension) devrait être proche de 1 (grâce à softmax).
assert output.shape == (batch_size, seq_len, hidden_dim), "La forme de la sortie est incorrecte!"
assert weights.shape == (batch_size, seq_len, seq_len), "La forme des poids d'attention est incorrecte!"

print("Les formes sont correctes pour ScaledDotProductAttention !")
print("Quelques poids d'attention pour le premier exemple, première requête :")
print(weights[0, 0, :])


--- Validation de ScaledDotProductAttention ---
Forme de l'entrée factice : torch.Size([2, 5, 64])
Forme de la sortie de l'attention : torch.Size([2, 5, 64])
Forme des poids d'attention : torch.Size([2, 5, 5])
Les formes sont correctes pour ScaledDotProductAttention !
Quelques poids d'attention pour le premier exemple, première requête :
tensor([0.1796, 0.1013, 0.2362, 0.2240, 0.2588], grad_fn=<SelectBackward0>)


## Tâche : Module d'Attention Multi-Têtes (Multi-Head Attention)

**Objectif :** Étendre le bloc à tête unique pour obtenir une fonctionnalité multi-têtes.

**Instructions :**
*   Implémentez `MultiHeadAttention` qui divise les *embeddings* en `num_heads`, applique l'attention par tête, puis concatène les résultats.
*   Incluez un *dropout* et des connexions résiduelles (residual connections).
*   Fournissez un exemple de passe avant (`forward`) montrant les formes d'entrée/sortie.
*   Fonctions à utiliser : `einops.rearrange` (facultatif, mais très utile !), `torch.reshape`, `torch.nn.Dropout`.

---

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
# Nous allons utiliser einops, une bibliothèque très pratique pour réorganiser les tenseurs.
# Si elle n'est pas installée, nous le faisons ici.
try:
    from einops import rearrange
except ImportError:
    !pip install einops
    from einops import rearrange


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout_rate=0.1):
        super(MultiHeadAttention, self).__init__()
        # d_model est la dimension totale de notre modèle (ex: 512, 768).
        # num_heads est le nombre de 'têtes' d'attention que nous voulons avoir.
        # Chaque tête apprendra une perspective différente des relations entre les mots.
        self.num_heads = num_heads
        self.d_model = d_model

        # La dimension de chaque tête. d_model doit être divisible par num_heads.
        assert d_model % num_heads == 0, "d_model doit être divisible par num_heads"
        self.d_k = d_model // num_heads # Dimension de la clé (K) et de la requête (Q) pour chaque tête.

        # Nous avons des projections linéaires pour Q, K, V pour TOUTES les têtes ensemble.
        # Ensuite, nous les diviserons en num_heads parties.
        self.query_projection = nn.Linear(d_model, d_model)
        self.key_projection = nn.Linear(d_model, d_model)
        self.value_projection = nn.Linear(d_model, d_model)

        # Le module d'attention par produit scalaire mis à l'échelle que nous avons défini précédemment.
        self.attention = ScaledDotProductAttention(self.d_k)

        # Couche linéaire finale pour recombiner les sorties de toutes les têtes.
        self.output_projection = nn.Linear(d_model, d_model)

        # Dropout pour aider à prévenir l'overfitting.
        self.dropout = nn.Dropout(dropout_rate)

        # Couche de normalisation (Layer Normalization) et connexion résiduelle seront ajoutées dans l'encodeur complet.
        # Ici, nous nous concentrons sur le module d'attention lui-même.

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        # 1. Projections linéaires pour Q, K, V (pour toutes les têtes).
        q = self.query_projection(query)
        k = self.key_projection(key)
        v = self.value_projection(value)

        # 2. Diviser les Q, K, V en 'num_heads' têtes et transposer pour avoir (batch_size, num_heads, seq_len, d_k).
        # einops.rearrange est très pratique ici pour remodeler et permuter les dimensions.
        # Avant: (batch_size, seq_len, d_model)
        # Après: (batch_size, num_heads, seq_len, d_k) où d_k = d_model / num_heads
        q = rearrange(q, 'b s (h d) -> b h s d', h=self.num_heads)
        k = rearrange(k, 'b s (h d) -> b h s d', h=self.num_heads)
        v = rearrange(v, 'b s (h d) -> b h s d', h=self.num_heads)

        # 3. Appliquer l'attention par produit scalaire à chaque tête.
        # Note: self.attention est un module ScaledDotProductAttention qui attend d_k comme d_model.
        # Nous passons donc q, k, v qui ont déjà la dimension d_k pour la dernière dimension.
        # Le masque doit être adapté : il doit être répliqué pour chaque tête.
        if mask is not None:
            # Ajout d'une dimension pour les têtes (num_heads) : (batch_size, 1, seq_len, seq_len)
            mask = mask.unsqueeze(1)

        # output_per_head aura la forme (batch_size, num_heads, seq_len, d_k)
        # weights_per_head aura la forme (batch_size, num_heads, seq_len, seq_len)
        output_per_head, weights_per_head = self.attention(q, k, v, mask=mask)

        # 4. Concaténer les sorties de toutes les têtes.
        # Nous inversons l'opération de réarrangement : de (batch_size, num_heads, seq_len, d_k)
        # à (batch_size, seq_len, d_model)
        concat_output = rearrange(output_per_head, 'b h s d -> b s (h d)')

        # 5. Appliquer une projection linéaire finale.
        output = self.output_projection(concat_output)

        # Appliquer le dropout (souvent utilisé après la projection finale ou la somme résiduelle)
        output = self.dropout(output)

        # Nous retournons l'output final et les poids d'attention (qui peuvent être moyennés ou traités autrement pour l'analyse).
        # Ici, nous retournons les poids de toutes les têtes tel quel.
        return output, weights_per_head


# --- Validation des formes avec des tenseurs factices pour MultiHeadAttention ---
print("\n--- Validation de MultiHeadAttention ---")

batch_size = 2
seq_len = 5
hidden_dim = 64  # La dimension totale du modèle
num_heads = 8    # Nombre de têtes. Doit diviser hidden_dim.

# Vérifions la condition de divisibilité
if hidden_dim % num_heads != 0:
    print(f"Attention: hidden_dim ({hidden_dim}) n'est pas divisible par num_heads ({num_heads}). Ajustement de num_heads à 4.")
    num_heads = 4 # On ajuste pour l'exemple si ce n'est pas le cas
    assert hidden_dim % num_heads == 0

dummy_input_mha = torch.randn(batch_size, seq_len, hidden_dim)

print(f"Forme de l'entrée factice pour MultiHeadAttention : {dummy_input_mha.shape}")

mha_module = MultiHeadAttention(d_model=hidden_dim, num_heads=num_heads)

output_mha, weights_mha = mha_module(dummy_input_mha, dummy_input_mha, dummy_input_mha)

print(f"Forme de la sortie de MultiHeadAttention : {output_mha.shape}")
print(f"Forme des poids d'attention de MultiHeadAttention : {weights_mha.shape}")

# Vérifions que les formes sont comme prévu :
# La sortie devrait avoir la forme (batch_size, seq_len, d_model)
# Les poids d'attention devraient être (batch_size, num_heads, seq_len, seq_len)
assert output_mha.shape == (batch_size, seq_len, hidden_dim), "La forme de la sortie MHA est incorrecte!"
assert weights_mha.shape == (batch_size, num_heads, seq_len, seq_len), "La forme des poids d'attention MHA est incorrecte!"

print("Les formes sont correctes pour MultiHeadAttention !")
print("Quelques poids d'attention pour la première tête, premier exemple, première requête :")
print(weights_mha[0, 0, 0, :])


--- Validation de MultiHeadAttention ---
Forme de l'entrée factice pour MultiHeadAttention : torch.Size([2, 5, 64])
Forme de la sortie de MultiHeadAttention : torch.Size([2, 5, 64])
Forme des poids d'attention de MultiHeadAttention : torch.Size([2, 8, 5, 5])
Les formes sont correctes pour MultiHeadAttention !
Quelques poids d'attention pour la première tête, premier exemple, première requête :
tensor([0.2219, 0.1795, 0.1912, 0.1937, 0.2138], grad_fn=<SelectBackward0>)
